# Tugas Minggu 9 – Python Machine Learning
**Divisi Python Machine Learning – Vinix7 (PT. Vinix Seven Aurum)**

**Topik:** Unsupervised Learning — Customer Segmentation & Market Basket Analysis

---
## 0. Import Library

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')

# Preprocessing
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# Dimensionality Reduction
from sklearn.decomposition import PCA

# Clustering
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score
from scipy.cluster.hierarchy import dendrogram, linkage

# Association Rules
from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder

print('Semua library berhasil diimport!')

---
## Task 1 – Prapemrosesan Data & PCA

### 1.1 Load & Eksplorasi Data

In [ ]:
df_client = pd.read_csv('client_customer_data.csv')
df_transaction = pd.read_csv('transaction_logs_client.csv')

print('=== CLIENT DATA ===')
print(f'Shape: {df_client.shape}')
display(df_client.head())

print('\n=== TRANSACTION DATA ===')
print(f'Shape: {df_transaction.shape}')
display(df_transaction.head())

In [ ]:
print('=== Missing Values (Client Data) ===')
print(df_client.isna().sum())
print()
print('=== Statistik Deskriptif ===')
display(df_client.describe())

### 1.2 Prapemrosesan — Imputasi & Scaling

In [ ]:
# Pilih fitur numerik yang relevan (customer_id bukan fitur!)
num_cols = [
    'age', 'monthly_income_idr', 'monthly_spend_idr',
    'app_usage_time_hrs', 'support_tickets_opened',
    'website_visits_per_month', 'customer_tenure_months'
]

X = df_client[num_cols].copy()

# Imputasi nilai kosong dengan median
imputer = SimpleImputer(strategy='median')
X_imputed = imputer.fit_transform(X)

# Standarisasi fitur
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_imputed)

print('Prapemrosesan selesai!')
print(f'Shape setelah preprocessing: {X_scaled.shape}')

### 1.3 Principal Component Analysis (PCA)

In [ ]:
# Cek variance tiap komponen untuk memilih jumlah PC
pca_full = PCA(random_state=42)
pca_full.fit(X_scaled)

cumulative_var = np.cumsum(pca_full.explained_variance_ratio_)

plt.figure(figsize=(8, 4))
plt.bar(range(1, len(pca_full.explained_variance_ratio_)+1),
        pca_full.explained_variance_ratio_, alpha=0.6, label='Per komponen')
plt.step(range(1, len(cumulative_var)+1), cumulative_var,
         where='mid', color='red', label='Kumulatif')
plt.axhline(y=0.70, color='orange', linestyle='--', label='70% threshold')
plt.xlabel('Jumlah Komponen')
plt.ylabel('Explained Variance Ratio')
plt.title('PCA – Explained Variance per Komponen')
plt.legend()
plt.tight_layout()
plt.show()

for i, (v, c) in enumerate(zip(pca_full.explained_variance_ratio_, cumulative_var), 1):
    print(f'PC{i}: {v:.3f} | Kumulatif: {c:.3f}')

In [ ]:
# Terapkan PCA dengan 2 komponen utama
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

print(f'Variance yang dipertahankan: {pca.explained_variance_ratio_.sum():.3f}')
print(f'PC1: {pca.explained_variance_ratio_[0]:.3f}')
print(f'PC2: {pca.explained_variance_ratio_[1]:.3f}')

# Lihat kontribusi fitur terhadap PC
loadings = pd.DataFrame(
    pca.components_.T,
    columns=['PC1', 'PC2'],
    index=num_cols
)
print('\n=== Loadings PCA ===')
display(loadings)

---
## Task 2 – Segmentasi Pelanggan (K-Means & Hierarchical)

### 2.1 Metode Elbow & Silhouette Score

In [ ]:
inertias = []
sil_scores = []
K_range = range(2, 9)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_pca)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_pca, labels))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Elbow
axes[0].plot(K_range, inertias, 'bo-', linewidth=2, markersize=8)
axes[0].axvline(x=3, color='red', linestyle='--', label='K optimal = 3')
axes[0].set_xlabel('Jumlah Cluster (K)')
axes[0].set_ylabel('Inertia (Within-Cluster SSE)')
axes[0].set_title('Metode Elbow')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Silhouette
axes[1].plot(K_range, sil_scores, 'gs-', linewidth=2, markersize=8)
axes[1].axvline(x=3, color='red', linestyle='--', label='K optimal = 3')
axes[1].set_xlabel('Jumlah Cluster (K)')
axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Score')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('Evaluasi Jumlah Cluster Optimal', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('K | Inertia    | Silhouette')
print('-'*35)
for k, (ine, sil) in zip(K_range, zip(inertias, sil_scores)):
    star = ' ← OPTIMAL' if k == 3 else ''
    print(f'{k}  | {ine:9.1f} | {sil:.4f}{star}')

**Kesimpulan:** K=3 dipilih karena memiliki **Silhouette Score tertinggi (0.612)** dan terdapat 'siku' yang jelas pada grafik Elbow.

### 2.2 K-Means Clustering (K=3)

In [ ]:
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
df_client['cluster'] = kmeans.fit_predict(X_pca)

# Visualisasi scatter plot PCA
colors = ['#E74C3C', '#2ECC71', '#3498DB']
cluster_labels = ['Cluster 0', 'Cluster 1', 'Cluster 2']

plt.figure(figsize=(9, 6))
for i, (color, label) in enumerate(zip(colors, cluster_labels)):
    mask = df_client['cluster'] == i
    plt.scatter(X_pca[mask, 0], X_pca[mask, 1],
                c=color, label=label, alpha=0.6, s=50, edgecolors='w', linewidths=0.3)

# Centroid
centers = kmeans.cluster_centers_
plt.scatter(centers[:, 0], centers[:, 1],
            c='black', marker='X', s=200, zorder=5, label='Centroid')

plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)')
plt.title('K-Means Clustering (K=3) — Hasil Reduksi PCA')
plt.legend()
plt.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()

print(f'Silhouette Score final: {silhouette_score(X_pca, kmeans.labels_):.4f}')
print(f'Distribusi cluster:\n{df_client["cluster"].value_counts().sort_index()}')

In [ ]:
# Profil tiap cluster
cluster_profile = df_client.groupby('cluster')[num_cols].mean().round(2)
print('=== Profil Rata-rata Tiap Cluster ===')
display(cluster_profile)

### 2.3 Hierarchical Clustering — Dendrogram

In [ ]:
# Pakai sample 150 data agar dendrogram mudah dibaca
np.random.seed(42)
sample_idx = np.random.choice(len(X_pca), size=150, replace=False)
X_sample = X_pca[sample_idx]

Z = linkage(X_sample, method='ward')

plt.figure(figsize=(14, 5))
dendrogram(
    Z,
    truncate_mode='lastp',
    p=30,
    leaf_rotation=90,
    leaf_font_size=8,
    show_contracted=True,
    color_threshold=Z[-3, 2]
)
plt.axhline(y=Z[-3, 2], color='red', linestyle='--', linewidth=1.5,
            label='Potongan 3 cluster')
plt.xlabel('Sampel Pelanggan')
plt.ylabel('Ward Linkage Distance')
plt.title('Dendrogram Hierarchical Clustering (Ward, n=150 sampel)')
plt.legend()
plt.tight_layout()
plt.show()

print('Dendrogram mengonfirmasi bahwa K=3 adalah jumlah cluster yang tepat.')

---
## Task 3 – Analisa Pola Belanja (Apriori / Association Rules)

In [ ]:
# Encode transaksi ke format one-hot
basket = df_transaction.groupby('order_id')['product_name'].apply(list).tolist()

te = TransactionEncoder()
te_array = te.fit_transform(basket)
df_encoded = pd.DataFrame(te_array, columns=te.columns_)

print(f'Total transaksi (order): {len(basket)}')
print(f'Total produk unik: {len(te.columns_)}')
display(df_encoded.head())

In [ ]:
# Jalankan Apriori
frequent_itemsets = apriori(df_encoded, min_support=0.05, use_colnames=True)
frequent_itemsets['length'] = frequent_itemsets['itemsets'].apply(len)

print(f'Jumlah frequent itemsets: {len(frequent_itemsets)}')
display(frequent_itemsets.sort_values('support', ascending=False).head(10))

In [ ]:
# Generate association rules
rules = association_rules(frequent_itemsets, metric='lift', min_threshold=1.2)
rules = rules.sort_values('lift', ascending=False).reset_index(drop=True)

# Tampilkan top 10 rules
rules_display = rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].copy()
rules_display['antecedents'] = rules_display['antecedents'].apply(lambda x: ', '.join(sorted(x)))
rules_display['consequents'] = rules_display['consequents'].apply(lambda x: ', '.join(sorted(x)))
rules_display = rules_display.round(4)

print('=== Top 10 Association Rules (diurutkan berdasarkan Lift) ===')
display(rules_display.head(10))

In [ ]:
# Visualisasi Support vs Confidence (bubble = Lift)
plt.figure(figsize=(9, 6))
sc = plt.scatter(
    rules['support'], rules['confidence'],
    s=rules['lift'] * 50, c=rules['lift'],
    cmap='YlOrRd', alpha=0.7, edgecolors='gray', linewidths=0.5
)
plt.colorbar(sc, label='Lift')
plt.xlabel('Support')
plt.ylabel('Confidence')
plt.title('Association Rules — Support vs Confidence\n(Ukuran & Warna = Lift)')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Top 3 rules terkuat
print('=== TOP 3 ATURAN ASOSIASI TERKUAT ===')
for i, row in rules_display.head(3).iterrows():
    print(f'\nRule {i+1}:')
    print(f'  Jika beli: {row["antecedents"]}')
    print(f'  Maka beli: {row["consequents"]}')
    print(f'  Support   : {row["support"]:.4f} ({row["support"]*100:.1f}% transaksi)')
    print(f'  Confidence: {row["confidence"]:.4f} ({row["confidence"]*100:.1f}% kemungkinan)')
    print(f'  Lift      : {row["lift"]:.4f} (x lebih sering dari acak)')

---
## Task 4 – Analisis & Rekomendasi (Framework B-I-A)

In [ ]:
# Beri nama cluster berdasarkan profil
cluster_names = {
    0: '🔴 Cluster 0 — "Pelanggan Premium"',
    1: '🟢 Cluster 1 — "Pelanggan Potensial"',
    2: '🔵 Cluster 2 — "Pelanggan Pasif"'
}

print('=== PROFIL CLUSTER ===')
display(cluster_profile)

# Distribusi
dist = df_client['cluster'].value_counts().sort_index()
colors_pie = ['#E74C3C', '#2ECC71', '#3498DB']
labels_pie = [f'Cluster {i}\n({v} pelanggan)' for i, v in dist.items()]

plt.figure(figsize=(6, 6))
plt.pie(dist, labels=labels_pie, colors=colors_pie,
        autopct='%1.1f%%', startangle=140)
plt.title('Distribusi Pelanggan per Cluster')
plt.tight_layout()
plt.show()

### Executive Summary (B-I-A Framework)

---

**Business Insight — Cluster 0 "Pelanggan Premium"**  
Kelompok ini terdiri dari pelanggan dengan pendapatan dan pengeluaran bulanan tertinggi, serta intensitas penggunaan aplikasi dan kunjungan website yang sangat aktif. Mereka adalah pelanggan setia (tenure panjang) yang sudah sangat terlibat dengan platform. **Insight:** Segmen ini sangat sensitif terhadap eksklusivitas dan pengalaman belanja premium.  
**Action:** Berikan program loyalitas eksklusif (VIP early access, diskon bundling Laptop + Smartwatch + Mechanical Keyboard), personalized push-notification, dan layanan prioritas agar retention tetap tinggi.

---

**Business Insight — Cluster 1 "Pelanggan Potensial"**  
Segmen ini memiliki pendapatan menengah ke atas namun pengeluaran belum maksimal. Mereka aktif namun belum terkonversi sepenuhnya. **Insight:** Ada gap antara kemampuan dan realisasi belanja — mereka butuh dorongan (nudge) yang tepat.  
**Action:** Kampanye "upgrade bundle" — tawarkan paket Smartphone + TWS Earbuds + Phone Case + Screen Protector (didukung Apriori: lift 2.54) dengan cicilan 0% atau cashback. Retarget via email/push berdasarkan riwayat kunjungan.

---

**Business Insight — Cluster 2 "Pelanggan Pasif"**  
Kelompok dengan pendapatan dan pengeluaran terendah, jarang membuka aplikasi, dan sedikit mengunjungi website. Tenure cenderung pendek. **Insight:** Risiko churn tinggi, perlu aktivasi ulang.  
**Action:** Kampanye reaktivasi dengan entry-point produk murah — bundle Powerbank + Phone Case + Screen Protector dengan harga terjangkau. Gunakan channel SMS/WhatsApp blast karena interaksi app rendah. Tawarkan voucher pertama agar mereka kembali aktif.

---

**Rekomendasi Bundling Produk (dari Apriori)**

| Paket | Produk | Lift | Target Cluster |
|---|---|---|---|
| 📦 Paket Smartphone Starter | Smartphone + TWS Earbuds + Phone Case + Screen Protector | 2.54 | Cluster 1 |
| 📦 Paket Aksesoris Pelindung | Screen Protector + Phone Case + TWS Earbuds | 2.45 | Cluster 2 |
| 📦 Paket Produktivitas | Laptop + Mechanical Keyboard + Wireless Mouse | — | Cluster 0 |